# 📓 Comprehensive Notebook Reference — Team BigBug, OctWave 3.0

> This document is a complete, chronological reference of **everything** we did, **every decision** we made, and **every finding** we discovered — from raw data loading to final submission generation. It is intended as the single source of truth for building the final consolidated Jupyter Notebook.

---

## Table of Contents

1. [Competition Context](#1-competition-context)
2. [Phase 1: Data Loading & Initial Inspection](#2-phase-1-data-loading--initial-inspection)
3. [Phase 2: Exploratory Data Analysis (EDA)](#3-phase-2-exploratory-data-analysis-eda)
4. [Phase 3: Data Cleaning & Validation](#4-phase-3-data-cleaning--validation)
5. [Phase 4: Feature Engineering](#5-phase-4-feature-engineering)
6. [Phase 5: Model Training & Hyperparameter Tuning](#6-phase-5-model-training--hyperparameter-tuning)
7. [Phase 6: Ensemble Construction](#7-phase-6-ensemble-construction)
8. [Phase 7: Advanced Ensemble (Stacking)](#8-phase-7-advanced-ensemble-stacking)
9. [Phase 8: Final Model Selection & Submission](#9-phase-8-final-model-selection--submission)
10. [Final Evaluation, Visualization & Submission Strategy](#10-final-evaluation-visualization--submission-strategy)
11. [Final Results Summary](#11-final-results-summary)

---

## 1. Competition Context

### Problem Statement
Build a binary classifier to detect **fraudulent credit card transactions** (`is_fraud = 0 or 1`) from a simulated dataset with extreme class imbalance.

### Key Constraints
- **Evaluation Metric:** F1-Score (balances Precision and Recall)
- **Timeline:** 9th August 2026 (12:00 NOON) → 11th August 2026 (11:59 PM)
- **Submission Limit:** Maximum 10 submissions per day
- **Final Evaluation:** Up to 2 submissions may be selected for the Private Leaderboard
- **External Data:** Strictly prohibited

### Dataset Statistics
| Property | Train (`train.csv`) | Test (`test.csv`) |
|---|---|---|
| Rows | 8,000 | 2,000 |
| Features | 9 + target (`is_fraud`) | 9 (no target) |
| File size | ~297 KB | ~70 KB |

### Raw Features (9 columns + target)
| Feature | Type | Range / Values | Description |
|---|---|---|---|
| `transaction_id` | Integer | 1 – 10,000 | Unique identifier (NOT a predictive feature) |
| `amount` | Float | 0.00 – 1,471.04 | Monetary value of the transaction |
| `transaction_hour` | Integer | 0 – 23 | Hour of the day |
| `merchant_category` | Categorical | Food, Clothing, Travel, Electronics, Grocery | Merchant type |
| `foreign_transaction` | Binary | 0, 1 | Was the transaction foreign? |
| `location_mismatch` | Binary | 0, 1 | Location mismatch flag |
| `device_trust_score` | Integer | 25 – 99 | Device trust rating |
| `velocity_last_24h` | Integer | 0 – 9 | Transactions in last 24 hours |
| `cardholder_age` | Integer | 18 – 69 | Age of the cardholder |
| `is_fraud` | Binary | 0, 1 | **Target variable** (train only) |

### Class Distribution (Target)
| Class | Count | Percentage |
|---|---|---|
| 0 (Legitimate) | 7,879 | 98.49% |
| 1 (Fraud) | 121 | 1.51% |

**Imbalance Ratio: 65:1** — This is extreme imbalance and the single most critical challenge.

---

## 2. Phase 1: Data Loading & Initial Inspection

**Source:** `src/data_processing/analyze_data.py`

### What We Did

### Checks Performed
1. **Shape verification:**
   - `train_df.shape` → (8000, 10) — 9 features + 1 target ✅
   - `test_df.shape` → (2000, 9) — 9 features, no target ✅

2. **Missing values check:**
   **Finding:** Zero missing values in both datasets. No imputation required.

3. **Data types inspection:**
   **Finding:** All columns have consistent data types between train and test. `merchant_category` is the only string/object column — all others are numeric (int64 or float64). ✅

4. **Descriptive statistics:**
   This gave us the ranges, means, and standard deviations for all numerical columns.

5. **Class balance check:**
   **Finding:** Only 1.51% of transactions are fraudulent (121 out of 8,000). This is a **65:1 imbalance ratio**.

6. **Merchant category distribution:**
   **Finding:** 5 unique categories — Food, Clothing, Travel, Electronics, Grocery. Roughly balanced distribution across categories.

---

## 3. Phase 2: Exploratory Data Analysis (EDA)

**Source:** `notebooks/01_EDA.ipynb`, `docs/data_cleaning/data_cleaning_plan.md`

### 3.1 Duplicate Analysis
- ✅ **No duplicate rows** in train or test
- ✅ **No duplicate `transaction_id`** values in either set
- ✅ **No overlapping `transaction_id`** between train and test
- **Action:** No deduplication required

### 3.2 Outlier Analysis (IQR Method)
| Feature | Outliers | % of Train |
|---|---|---|
| `amount` | 401 | 5.01% |
| `velocity_last_24h` | 41 | 0.51% |
| `transaction_hour` | 0 | 0.00% |
| `device_trust_score` | 0 | 0.00% |
| `cardholder_age` | 0 | 0.00% |

**Critical Decision:** We decided **NOT to remove outliers**. In fraud detection, outliers are **signal, not noise**. High-amount transactions and high-velocity transactions are correlated with fraud. Removing them would destroy predictive power.

### 3.3 Zero/Edge Value Analysis
| Feature | Zeros | Notes |
|---|---|---|
| `amount` | 1 | A $0.00 transaction — could be a test/probe transaction (common fraud pattern) |
| `transaction_hour` | 341 | Valid — hour `0` means midnight |
| `velocity_last_24h` | 1,120 | Valid — 0 prior transactions in 24h |

**Decision:** All zeros are valid and meaningful. The single `amount = 0` record was kept (zero-dollar test charges are a known fraud pattern). No negative values found anywhere.

### 3.4 Fraud Signal Discovery
This was the most critical part of EDA — understanding which features carry the strongest fraud signal.

| Signal | Fraud Rate | Baseline (1.51%) | Uplift |
|---|---|---|---|
| `location_mismatch = 1` | 8.24% | 1.51% | 🔴 **5.5×** |
| `foreign_transaction = 1` | 8.12% | 1.51% | 🔴 **5.4×** |
| `device_trust_score` 20-40 | 6.04% | 1.51% | 🔴 **4.0×** |
| Night transactions (0-5h) | 4.91% | 1.51% | 🟠 **3.3×** |
| `amount` > 500 | 3.04% | 1.51% | 🟡 **2.0×** |
| High `velocity_last_24h` | corr: +0.110 | — | 🟡 Mild |
| `cardholder_age` | corr: +0.000 | — | ⚪ None |

### 3.5 Correlation Analysis with `is_fraud`
| Feature | Correlation |
|---|---|
| `foreign_transaction` | **+0.179** |
| `location_mismatch` | **+0.168** |
| `device_trust_score` | **−0.138** |
| `transaction_hour` | **−0.135** |
| `velocity_last_24h` | +0.110 |
| `amount` | +0.034 |
| `cardholder_age` | +0.000 |

**Key Insight:** `cardholder_age` has zero correlation with fraud — it provides no predictive signal at all. However, we kept it in the feature set because tree-based models can safely ignore irrelevant features, and removing it could hurt models that find subtle interactions.

### 3.6 Train vs Test Distribution Consistency
| Feature | Train Mean | Test Mean | Drift? |
|---|---|---|---|
| `amount` | ~175 | ~178 | ✅ No drift |
| `device_trust_score` | ~62 | ~61 | ✅ No drift |

**Finding:** All feature distributions are consistent between train and test. No distribution drift correction needed.

### 3.7 Critical Visualizations Performed
The following visualizations were instrumental in discovering the patterns above and must be reproduced in the final notebook:
- **Numerical feature histograms**
- **Side-by-side boxplots** (Fraud vs Legitimate) for continuous features
- **Merchant category distribution**
- **Correlation heatmap** (full feature matrix)
- **Deep dive plots** for `transaction_hour`, `device_trust_score`, `amount`, and `velocity_last_24h` split by fraud class
- **Feature interaction plot** (`foreign_transaction` × `location_mismatch`)
- **Pairplot** colored by fraud class
- **Train vs Test distribution overlay** plots

---

## 4. Phase 3: Data Cleaning & Validation

**Source:** `src/data_processing/preprocess.py` (lines 43-105)

### What We Did (Step by Step)

#### Step 1: Load Raw Data

#### Step 2: Preserve Test IDs
**Rationale:** `transaction_id` is needed for the final `submission.csv` but has zero predictive value. We save it separately before dropping it from the feature matrix.

#### Step 3: Drop Identifier Column

#### Step 4: Separate Target Variable

#### Step 5: Combine Train & Test for Consistent Engineering
**Rationale:** By combining them before feature engineering, we guarantee that one-hot encoding produces the exact same columns for both sets (no missing category columns in test).

#### Step 6: Sanity Checks (Post-Processing)
**These 4 assertions guarantee:**
1. No null values were introduced during feature engineering
2. No null values exist in test
3. The target column is present in train
4. The target column has NOT leaked into test (critical data leakage check)

---

## 5. Phase 4: Feature Engineering

**Source:** `src/data_processing/preprocess.py` → `engineer_features()` function (lines 11-41)

Starting from **9 raw features** (after dropping `transaction_id`), we engineered **16 additional features** for a total of **23 model-ready features** (excluding the target).

### 5.1 Time-Based Risk Features

#### `is_night_transaction` (Binary)
**Rationale:** EDA showed nighttime transactions (midnight–5am) have a **3.3× higher fraud rate** (4.91% vs 1.51% baseline).

#### `time_of_day_category` (Categorical → One-Hot Encoded)
**Rationale:** Provides coarser time granularity that tree-based models can use as clean decision boundaries instead of arbitrary hour integers.
**Result:** This later gets one-hot encoded into 4 binary columns: `time_of_day_category_Night`, `time_of_day_category_Morning`, `time_of_day_category_Afternoon`, `time_of_day_category_Evening`.

### 5.2 Trust and Anomaly Ratios

#### `amount_to_trust_ratio` (Continuous)
**Rationale:** Fraudsters attempt high-value transactions on low-trust devices. This ratio captures that interaction. The `1e-6` epsilon prevents division by zero (though the minimum trust score is 25, so it's a safety measure).

#### `amount_velocity_ratio` (Continuous)
**Rationale:** Represents the average spend per recent transaction. High velocity coupled with large amounts is a classic indicator of account takeover. The `+1` prevents division by zero (velocity can be 0).

### 5.3 High-Risk Location Flags

#### `is_high_risk_location` (Binary)
**Rationale:** Both `foreign_transaction` and `location_mismatch` individually correspond to an ~8% fraud rate (5.4× and 5.5× uplift). This feature flags ANY location anomaly.

#### `location_anomaly_score` (Integer, 0-2)
**Rationale:** Captures the *severity* of location anomalies. A score of 2 (both foreign AND location mismatch) is a much stronger fraud signal than either alone.

### 5.4 Strategic Binning

#### `is_high_amount` (Binary)
**Rationale:** EDA showed the fraud rate doubles for transactions above $500 (3.04% vs 1.51% baseline).

#### `is_low_trust` (Binary)
**Rationale:** The fraud rate is approximately 4× higher (6.04%) when the device trust score is below 40.

### 5.5 Categorical Encoding

#### One-Hot Encoding
**Decision: `drop_first=False`** — We chose NOT to drop the first category. While dropping one is standard for linear models (to avoid multicollinearity), our primary models are tree-based (XGBoost, CatBoost, AdaBoost), which are immune to multicollinearity. Keeping all categories preserves full information.

**Result:** `merchant_category` → 5 binary columns; `time_of_day_category` → 4 binary columns.

#### Boolean to Integer Conversion
**Rationale:** `pd.get_dummies()` creates boolean columns. Some models (especially CatBoost) can be sensitive to data types, so we convert bools to integers for universal compatibility.

### 5.6 Split Back & Save

### 5.7 Preserved Raw Features
**Critical Decision:** We kept the original numerical columns (`amount`, `transaction_hour`, `device_trust_score`, `velocity_last_24h`, `cardholder_age`) intact alongside their engineered counterparts. Tree-based models benefit from having access to both raw distributions AND engineered signals.

### 5.8 Final Feature Count Summary
| Category | Features | Count |
|---|---|---|
| Raw numerical | `amount`, `transaction_hour`, `foreign_transaction`, `location_mismatch`, `device_trust_score`, `velocity_last_24h`, `cardholder_age` | 7 |
| Engineered numerical | `is_night_transaction`, `amount_to_trust_ratio`, `amount_velocity_ratio`, `is_high_risk_location`, `location_anomaly_score`, `is_high_amount`, `is_low_trust` | 7 |
| One-Hot: merchant_category | `Food`, `Clothing`, `Travel`, `Electronics`, `Grocery` | 5 |
| One-Hot: time_of_day_category | `Night`, `Morning`, `Afternoon`, `Evening` | 4 |
| **Total features** | | **23** |

**Output Files:**
- `data/processed/train_engineered.csv` → 23 features + `is_fraud` target (24 columns)
- `data/processed/test_engineered.csv` → 23 features (no target)
- `data/processed/test_transaction_ids.csv` → Transaction IDs for submission

### 5.9 Feature Scaling Rationale
**Source:** `notebooks/02_Data_Preprocessing.ipynb`

Scaling was intentionally **deferred** to the modeling pipeline rather than being applied during data preprocessing.

| Model Type | Scaling Needed? | Reason |
|---|---|---|
| Random Forest, XGBoost, LightGBM, CatBoost | ❌ No | Tree-based models split on feature values and are scale-invariant. |
| Logistic Regression, SVM, Neural Networks | ✅ Yes | Distance/gradient-based models are highly sensitive to feature scales. |

**Decision:** Because our primary models are tree-based, keeping raw feature values preserves interpretability and allows tree-based models to access natural distributions. Models that require scaling will apply `StandardScaler` inside their own Scikit-Learn pipelines.

---

In [ ]:
import pandas as pd
import numpy as np
import os

def load_data(base_dir):
    raw_dir = os.path.join(base_dir, "data", "raw")
    train = pd.read_csv(os.path.join(raw_dir, "train.csv"))
    test = pd.read_csv(os.path.join(raw_dir, "test.csv"))
    return train, test

def engineer_features(df):
    """Applies feature engineering logic to a dataframe."""
    
    # 1. Time-Based Risk Features
    df['is_night_transaction'] = df['transaction_hour'].apply(lambda x: 1 if 0 <= x <= 5 else 0)
    
    # Binning time of day
    def get_time_category(hour):
        if 0 <= hour <= 5: return 'Night'
        elif 6 <= hour <= 11: return 'Morning'
        elif 12 <= hour <= 17: return 'Afternoon'
        else: return 'Evening'
    
    df['time_of_day_category'] = df['transaction_hour'].apply(get_time_category)
    
    # 2. Trust and Anomaly Ratios
    # Add small epsilon to denominator to prevent division by zero (trust_score min is 25, so no div 0 anyway)
    df['amount_to_trust_ratio'] = df['amount'] / (df['device_trust_score'] + 1e-6)
    
    # +1 to avoid div by zero (velocity can be 0)
    df['amount_velocity_ratio'] = df['amount'] / (df['velocity_last_24h'] + 1)
    
    # 3. High-Risk Location Flags
    df['is_high_risk_location'] = ((df['foreign_transaction'] == 1) | (df['location_mismatch'] == 1)).astype(int)
    df['location_anomaly_score'] = df['foreign_transaction'] + df['location_mismatch']
    
    # 4. Strategic Binning
    df['is_high_amount'] = (df['amount'] > 500).astype(int)
    df['is_low_trust'] = (df['device_trust_score'] < 40).astype(int)
    
    return df

def preprocess_and_save():
    print("Starting Feature Engineering Pipeline...")
    
    base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..'))
    processed_dir = os.path.join(base_dir, "data", "processed")
    os.makedirs(processed_dir, exist_ok=True)
    
    train, test = load_data(base_dir)
    print(f"Loaded raw data. Train shape: {train.shape}, Test shape: {test.shape}")
    
    # Save test IDs
    test_ids = test['transaction_id']
    test_ids.to_csv(os.path.join(processed_dir, "test_transaction_ids.csv"), index=False)
    
    # Drop IDs from feature sets
    train.drop(columns=['transaction_id'], inplace=True)
    test.drop(columns=['transaction_id'], inplace=True)
    
    # Separate target
    y_train = train['is_fraud']
    train.drop(columns=['is_fraud'], inplace=True)
    
    # Combine for consistent feature engineering
    combined = pd.concat([train, test], axis=0)
    
    print("Engineering features...")
    combined = engineer_features(combined)
    
    # One-Hot Encoding
    print("Applying One-Hot Encoding...")
    cat_cols = ['merchant_category', 'time_of_day_category']
    combined_encoded = pd.get_dummies(combined, columns=cat_cols, drop_first=False)
    
    # Convert booleans to integers
    for col in combined_encoded.columns:
        if combined_encoded[col].dtype == 'bool':
            combined_encoded[col] = combined_encoded[col].astype(int)
            
    # Split back
    train_engineered = combined_encoded.iloc[:len(train)].copy()
    test_engineered = combined_encoded.iloc[len(train):].copy()
    
    # Add target back to train
    train_engineered['is_fraud'] = y_train.values
    
    # Save
    print("Saving engineered datasets...")
    train_out = os.path.join(processed_dir, "train_engineered.csv")
    test_out = os.path.join(processed_dir, "test_engineered.csv")
    
    train_engineered.to_csv(train_out, index=False)
    test_engineered.to_csv(test_out, index=False)
    
    print(f"Engineered Train shape: {train_engineered.shape}")
    print(f"Engineered Test shape: {test_engineered.shape}")
    
    # Sanity checks
    assert train_engineered.isnull().sum().sum() == 0, "Null values found in train!"
    assert test_engineered.isnull().sum().sum() == 0, "Null values found in test!"
    assert 'is_fraud' in train_engineered.columns, "is_fraud missing from train!"
    assert 'is_fraud' not in test_engineered.columns, "is_fraud found in test!"
    
    print("Pipeline completed successfully!")

if __name__ == "__main__":
    preprocess_and_save()


## 6. Phase 5: Model Training & Hyperparameter Tuning

**Source:** `src/modeling/train.py`

### 6.1 Strategy Overview
- **Objective:** Maximize F1-Score
- **Tuning Engine:** Optuna (Bayesian Optimization), 20 trials per model, 10-minute timeout per model
- **Validation:** 5-Fold Stratified Cross-Validation (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`)
- **Imbalance Handling:** Algorithmic class weights (NO SMOTE / NO resampling)

### 6.2 Class Imbalance Strategy
**Decision: Use native algorithmic class weights instead of SMOTE.**

| Algorithm Family | Weight Parameter | How It Works |
|---|---|---|
| XGBoost / LightGBM | `scale_pos_weight = 65.12` | Calculated as `count(negative) / count(positive)` = 7879 / 121 |
| CatBoost | `auto_class_weights = 'Balanced'` | CatBoost internally calculates balanced weights |
| Scikit-Learn (RF, ExtraTrees, LR, SVC) | `class_weight = 'balanced'` | Scikit-learn adjusts weights inversely proportional to class frequencies |
| AdaBoost, GradientBoosting | No native weight param | These models rely on their sequential boosting nature to focus on misclassified (minority) samples |
| KNN, GaussianNB, MLP | No class weight | These models don't support class weights natively |

**Why no SMOTE?** The native algorithmic weights pushed F1 scores above 0.99 without any data resampling. SMOTE would introduce synthetic samples that could distort decision boundaries, especially risky for a competition where the test distribution must match the training distribution exactly.

### 6.3 Feature Scaling Strategy
**Decision: Scale only for models that require it.**


- **Tree-based models (XGBoost, CatBoost, RF, etc.):** No scaling needed — trees split on feature values, so scale doesn't matter.
- **Distance/gradient-based models (LR, SVC, KNN, GNB, MLP):** `StandardScaler` applied via a `Pipeline` to prevent data leakage during cross-validation (the scaler is fit only on the training folds, not the validation fold).

### 6.4 The 12 Models Evaluated

Every model was tuned with Optuna (20 trials) and evaluated with 5-Fold Stratified CV:

#### Model 1: Logistic Regression
- Scaled with `StandardScaler`
- **Result:** F1 = 0.8077, Precision = 0.6862, Recall = 0.9837
- **Insight:** Very high recall (caught 98.37% of fraud) but too many false positives

#### Model 2: Random Forest
- **Result:** F1 = 0.9195, Precision = 0.9913, Recall = 0.8593

#### Model 3: ExtraTrees
- **Result:** F1 = 0.8113, Precision = 0.8720, Recall = 0.7677

#### Model 4: XGBoost
- **Result:** F1 = 0.9874, Precision = 1.0000, Recall = 0.9753
- **Best params:** `learning_rate=0.151, max_depth=5`

#### Model 5: LightGBM
- **Result:** F1 = 0.9785, Precision = 1.0000, Recall = 0.9583
- **Best params:** `learning_rate=0.110, max_depth=8`

#### Model 6: CatBoost
- **Result:** F1 = 0.9838, Precision = 0.9769, Recall = 0.9917
- **Best params:** `learning_rate=0.081, depth=6`

#### Model 7: AdaBoost
- **Result:** F1 = 0.9957, Precision = 1.0000, Recall = 0.9917 🏆
- **Best params:** `n_estimators=124, learning_rate=0.669`
- **Top single model!**

#### Model 8: Gradient Boosting
- **Result:** F1 = 0.9791, Precision = 0.9917, Recall = 0.9673

#### Model 9: SVC (Support Vector Classifier)
- Scaled with `StandardScaler`
- **Result:** F1 = 0.8205, Precision = 0.8234, Recall = 0.8253

#### Model 10: KNN (K-Nearest Neighbors)
- Scaled with `StandardScaler`
- **Result:** F1 = 0.5274, Precision = 0.7740, Recall = 0.4110
- **Worst performer after GNB** — distance-based models struggle with high-dimensional sparse features

#### Model 11: Gaussian Naive Bayes
- Scaled with `StandardScaler`
- **Result:** F1 = 0.3021, Precision = 0.1852, Recall = 0.8263
- **Worst performer** — the Gaussian assumption is violated by the binary/engineered features

#### Model 12: MLP Classifier (Neural Network)
- Scaled with `StandardScaler`
- **Result:** F1 = 0.8330, Precision = 0.8246, Recall = 0.8427

### 6.5 Final Model Leaderboard

| Rank | Model | F1-Score | Precision | Recall | PR-AUC |
|:---:|---|:---:|:---:|:---:|:---:|
| 🥇 | **AdaBoost** | **0.9957** | 1.0000 | 0.9917 | 0.9994 |
| 🥈 | **XGBoost** | **0.9874** | 1.0000 | 0.9753 | 0.9991 |
| 🥉 | **CatBoost** | **0.9838** | 0.9769 | 0.9917 | 0.9997 |
| 4 | Gradient Boosting | 0.9791 | 0.9917 | 0.9673 | 0.9918 |
| 5 | LightGBM | 0.9785 | 1.0000 | 0.9583 | 0.9916 |
| 6 | Random Forest | 0.9195 | 0.9913 | 0.8593 | 0.9675 |
| 7 | MLPClassifier | 0.8330 | 0.8246 | 0.8427 | 0.9052 |
| 8 | SVC | 0.8205 | 0.8234 | 0.8253 | 0.8735 |
| 9 | ExtraTrees | 0.8113 | 0.8720 | 0.7677 | 0.8884 |
| 10 | Logistic Regression | 0.8077 | 0.6862 | 0.9837 | 0.8919 |
| 11 | KNN | 0.5274 | 0.7740 | 0.4110 | 0.6045 |
| 12 | Gaussian Naive Bayes | 0.3021 | 0.1852 | 0.8263 | 0.3863 |

### 6.6 Key Takeaways from Model Training
1. **Boosting algorithms dominate** — all 5 top models are gradient boosting variants
2. **AdaBoost was the surprise champion** — despite being the oldest boosting algorithm, it achieved the best single-model F1 (0.9957) with perfect precision
3. **XGBoost achieved perfect Precision** (100%) — every transaction it flagged was actually fraud
4. **CatBoost achieved the highest Recall** (99.17%) — it caught nearly every fraudulent transaction
5. **Non-boosting models severely underperformed** — the gap between boosting (>0.97) and non-boosting (<0.92) is massive

### 6.7 Artifact Storage
For every model, we saved:
- **Model file:** `models/<model_name>/best_model.pkl` (pickle serialized)
- **Predictions:** `outputs/<model_name>/predictions.csv`
- **Central log:** `outputs/model_results_log.csv` (all metrics + hyperparameters)

---

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import pickle
import warnings
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import optuna

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def load_data():
    base_dir = r"c:\Users\ADMIN\Desktop\BigBug_Octwave"
    processed_dir = os.path.join(base_dir, "data", "processed")
    train = pd.read_csv(os.path.join(processed_dir, "train_engineered.csv"))
    test = pd.read_csv(os.path.join(processed_dir, "test_engineered.csv"))
    test_ids = pd.read_csv(os.path.join(processed_dir, "test_transaction_ids.csv"))
    return train, test, test_ids, base_dir

def init_logger(outputs_dir):
    log_path = os.path.join(outputs_dir, "model_results_log.csv")
    if not os.path.exists(log_path):
        with open(log_path, 'w') as f:
            f.write("Model_Name,CV_Accuracy,CV_F1_Score,CV_PR_AUC,CV_Precision,CV_Recall,Best_Parameters,Model_File_Path,Output_Predictions_Path\n")
    return log_path

def log_result(log_path, results):
    row = f"{results['Model_Name']},{results['CV_Accuracy']:.4f},{results['CV_F1_Score']:.4f},{results['CV_PR_AUC']:.4f}," \
          f"{results['CV_Precision']:.4f},{results['CV_Recall']:.4f},\"{str(results['Best_Parameters']).replace('\"','\'')}\"," \
          f"{results['Model_File_Path']},{results['Output_Predictions_Path']}\n"
    with open(log_path, 'a') as f:
        f.write(row)

# Define Model Objectives for Optuna
def get_objectives(X, y):
    pos_weight = (len(y) - sum(y)) / sum(y) # scale_pos_weight
    
    def get_cv_score(model, X, y, needs_scaling=False):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        if needs_scaling:
            model = Pipeline([('scaler', StandardScaler()), ('clf', model)])
        
        scores = cross_validate(model, X, y, cv=cv, scoring=['f1', 'average_precision', 'accuracy', 'precision', 'recall'], n_jobs=-1, error_score=0.0)
        return scores['test_f1'].mean()

    objectives = {
        'LogisticRegression': lambda trial: get_cv_score(
            LogisticRegression(
                C=trial.suggest_float('C', 1e-4, 10.0, log=True),
                class_weight='balanced',
                random_state=42,
                max_iter=1000
            ), X, y, needs_scaling=True),
            
        'RandomForest': lambda trial: get_cv_score(
            RandomForestClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                max_depth=trial.suggest_int('max_depth', 3, 10),
                class_weight='balanced',
                random_state=42,
                n_jobs=1
            ), X, y, needs_scaling=False),
            
        'ExtraTrees': lambda trial: get_cv_score(
            ExtraTreesClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                max_depth=trial.suggest_int('max_depth', 3, 15),
                class_weight='balanced',
                random_state=42,
                n_jobs=1
            ), X, y, needs_scaling=False),
            
        'XGBoost': lambda trial: get_cv_score(
            XGBClassifier(
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                max_depth=trial.suggest_int('max_depth', 3, 9),
                scale_pos_weight=pos_weight,
                eval_metric='logloss',
                random_state=42,
                n_jobs=1
            ), X, y, needs_scaling=False),
            
        'LightGBM': lambda trial: get_cv_score(
            LGBMClassifier(
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                max_depth=trial.suggest_int('max_depth', 3, 9),
                scale_pos_weight=pos_weight,
                random_state=42,
                n_jobs=1,
                verbose=-1
            ), X, y, needs_scaling=False),
            
        'CatBoost': lambda trial: get_cv_score(
            CatBoostClassifier(
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                depth=trial.suggest_int('depth', 4, 8),
                auto_class_weights='Balanced',
                random_state=42,
                verbose=0,
                thread_count=1
            ), X, y, needs_scaling=False),
            
        'AdaBoost': lambda trial: get_cv_score(
            AdaBoostClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 1.0, log=True),
                random_state=42
            ), X, y, needs_scaling=False),
            
        'GradientBoosting': lambda trial: get_cv_score(
            GradientBoostingClassifier(
                n_estimators=trial.suggest_int('n_estimators', 50, 200),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                max_depth=trial.suggest_int('max_depth', 3, 8),
                random_state=42
            ), X, y, needs_scaling=False),
            
        'SVC': lambda trial: get_cv_score(
            SVC(
                C=trial.suggest_float('C', 0.1, 10.0, log=True),
                class_weight='balanced',
                probability=True,
                random_state=42
            ), X, y, needs_scaling=True),
            
        'KNN': lambda trial: get_cv_score(
            KNeighborsClassifier(
                n_neighbors=trial.suggest_int('n_neighbors', 3, 15),
                weights=trial.suggest_categorical('weights', ['uniform', 'distance'])
            ), X, y, needs_scaling=True),
            
        'GaussianNB': lambda trial: get_cv_score(
            GaussianNB(
                var_smoothing=trial.suggest_float('var_smoothing', 1e-9, 1e-2, log=True)
            ), X, y, needs_scaling=True),
            
        'MLPClassifier': lambda trial: get_cv_score(
            MLPClassifier(
                hidden_layer_sizes=trial.suggest_categorical('hidden_layer_sizes', [(50,), (100,), (50, 50)]),
                alpha=trial.suggest_float('alpha', 1e-4, 1e-1, log=True),
                random_state=42,
                max_iter=500
            ), X, y, needs_scaling=True)
    }
    return objectives, pos_weight

def get_best_model_instance(model_name, best_params, pos_weight):
    needs_scaling = model_name in ['LogisticRegression', 'SVC', 'KNN', 'GaussianNB', 'MLPClassifier']
    
    if model_name == 'LogisticRegression': model = LogisticRegression(**best_params, class_weight='balanced', random_state=42, max_iter=1000)
    elif model_name == 'RandomForest': model = RandomForestClassifier(**best_params, class_weight='balanced', random_state=42)
    elif model_name == 'ExtraTrees': model = ExtraTreesClassifier(**best_params, class_weight='balanced', random_state=42)
    elif model_name == 'XGBoost': model = XGBClassifier(**best_params, scale_pos_weight=pos_weight, eval_metric='logloss', random_state=42)
    elif model_name == 'LightGBM': model = LGBMClassifier(**best_params, scale_pos_weight=pos_weight, random_state=42, verbose=-1)
    elif model_name == 'CatBoost': model = CatBoostClassifier(**best_params, auto_class_weights='Balanced', random_state=42, verbose=0)
    elif model_name == 'AdaBoost': model = AdaBoostClassifier(**best_params, random_state=42)
    elif model_name == 'GradientBoosting': model = GradientBoostingClassifier(**best_params, random_state=42)
    elif model_name == 'SVC': model = SVC(**best_params, class_weight='balanced', probability=True, random_state=42)
    elif model_name == 'KNN': model = KNeighborsClassifier(**best_params)
    elif model_name == 'GaussianNB': model = GaussianNB(**best_params)
    elif model_name == 'MLPClassifier': model = MLPClassifier(**best_params, random_state=42, max_iter=500)
    
    if needs_scaling:
        model = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    
    return model

def train_and_log_models():
    print("Starting Exhaustive Modeling Phase...")
    train, test, test_ids, base_dir = load_data()
    
    X = train.drop(columns=['is_fraud'])
    y = train['is_fraud']
    X_test = test
    
    models_dir = os.path.join(base_dir, "models")
    outputs_dir = os.path.join(base_dir, "outputs")
    os.makedirs(models_dir, exist_ok=True)
    os.makedirs(outputs_dir, exist_ok=True)
    
    log_path = init_logger(outputs_dir)
    objectives, pos_weight = get_objectives(X, y)
    
    N_TRIALS = 20
    
    for model_name, objective_func in objectives.items():
        print(f"\n--- Optimizing {model_name} ---")
        
        # Create nested dirs
        m_dir = os.path.join(models_dir, model_name.lower())
        o_dir = os.path.join(outputs_dir, model_name.lower())
        os.makedirs(m_dir, exist_ok=True)
        os.makedirs(o_dir, exist_ok=True)
        
        model_file = os.path.join(m_dir, "best_model.pkl")
        preds_file = os.path.join(o_dir, "predictions.csv")
        
        # Optuna Study
        study = optuna.create_study(direction="maximize")
        study.optimize(objective_func, n_trials=N_TRIALS, timeout=600) # max 10 mins per model
        
        best_params = study.best_params
        print(f"Best Params: {best_params}")
        
        # Final evaluation of best params
        final_model = get_best_model_instance(model_name, best_params, pos_weight)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = cross_validate(final_model, X, y, cv=cv, scoring=['f1', 'average_precision', 'accuracy', 'precision', 'recall'], n_jobs=-1)
        
        # Train on full data & save
        final_model.fit(X, y)
        with open(model_file, 'wb') as f:
            pickle.dump(final_model, f)
            
        # Predict on test
        preds = final_model.predict(X_test)
        sub = pd.DataFrame({'transaction_id': test_ids['transaction_id'], 'is_fraud': preds})
        sub.to_csv(preds_file, index=False)
        
        # Log
        results = {
            'Model_Name': model_name,
            'CV_Accuracy': scores['test_accuracy'].mean(),
            'CV_F1_Score': scores['test_f1'].mean(),
            'CV_PR_AUC': scores['test_average_precision'].mean(),
            'CV_Precision': scores['test_precision'].mean(),
            'CV_Recall': scores['test_recall'].mean(),
            'Best_Parameters': best_params,
            'Model_File_Path': f"models/{model_name.lower()}/best_model.pkl",
            'Output_Predictions_Path': f"outputs/{model_name.lower()}/predictions.csv"
        }
        log_result(log_path, results)
        print(f"Finished {model_name}. F1: {results['CV_F1_Score']:.4f}")

if __name__ == "__main__":
    train_and_log_models()


## 7. Phase 6: Ensemble Construction (3-Model Soft Voting)

**Source:** `src/modeling/ensemble.py`

### 7.1 Strategy
Combine the top 3 models (AdaBoost, XGBoost, CatBoost) into an Optuna-weighted Soft Voting Ensemble.

**Why only 3 models?** These 3 represent the clear top tier (F1 > 0.98). Adding weaker models (LightGBM at 0.9785, GradientBoosting at 0.9791) would dilute the ensemble quality without adding meaningful diversity.

### 7.2 Step-by-Step Process

#### Step 1: Load Pre-Trained Models

#### Step 2: Generate Out-Of-Fold (OOF) Probabilities
**Why OOF?** If we used regular training predictions to optimize weights, the weights would overfit to training data. OOF predictions are generated by models that never saw the data they're predicting on — this gives us an unbiased estimate.

#### Step 3: Optimize Weights with Optuna (100 Trials)
**Note:** Optuna ran 100 trials (not 20 like base models) because weight optimization is computationally cheap — it's just weighted averaging of pre-computed probabilities.

#### Step 4: Optimal Weights Found
| Model | Weight | Trust % |
|---|---|---|
| **XGBoost** | 0.423 | 42.3% |
| **CatBoost** | 0.292 | 29.2% |
| **AdaBoost** | 0.284 | 28.4% |

**Interesting finding:** Despite AdaBoost being the best single model (F1=0.9957), Optuna assigned it the lowest weight. This is because XGBoost provides the most *complementary* signal — it catches different fraud patterns than the other two.

#### Step 5: Build VotingClassifier & Train on Full Data

#### Step 6: Decision Threshold
**Decision:** We used the standard **0.5 threshold**. This was validated to give perfect precision (zero false alarms) with only 1 missed fraud case.

### 7.3 Ensemble Results
| Metric | Score |
|---|---|
| **F1-Score** | **0.9959** |
| **Precision** | **1.0000** |
| **Recall** | **0.9917** |
| **PR-AUC** | **0.9991** |
| **Accuracy** | **0.9999** |

**The ensemble marginally outperformed AdaBoost alone** (0.9959 vs 0.9957). The improvement is tiny but the ensemble provides better generalization insurance for the private leaderboard.

---

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import optuna
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, accuracy_score
from sklearn.ensemble import VotingClassifier

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def load_data():
    base_dir = r"c:\Users\ADMIN\Desktop\BigBug_Octwave"
    processed_dir = os.path.join(base_dir, "data", "processed")
    train = pd.read_csv(os.path.join(processed_dir, "train_engineered.csv"))
    test = pd.read_csv(os.path.join(processed_dir, "test_engineered.csv"))
    test_ids = pd.read_csv(os.path.join(processed_dir, "test_transaction_ids.csv"))
    return train, test, test_ids, base_dir

def load_models(base_dir):
    models = {}
    for name in ['adaboost', 'xgboost', 'catboost']:
        path = os.path.join(base_dir, "models", name, "best_model.pkl")
        with open(path, 'rb') as f:
            models[name] = pickle.load(f)
    return models

def get_oof_predictions(models, X, y):
    """Generate Out-Of-Fold probability predictions for each model to quickly tune weights."""
    print("Generating Out-Of-Fold probabilities (This might take a minute)...")
    oof_probs = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    for name, model in models.items():
        print(f"  Generating OOF for {name}...")
        # Get probability of class 1
        probs = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
        oof_probs[name] = probs
    return oof_probs

def optimize_weights(oof_probs, y):
    print("Optimizing Ensemble weights with Optuna...")
    
    def objective(trial):
        w_ada = trial.suggest_float('w_ada', 0.0, 1.0)
        w_xgb = trial.suggest_float('w_xgb', 0.0, 1.0)
        w_cat = trial.suggest_float('w_cat', 0.0, 1.0)
        
        total = w_ada + w_xgb + w_cat
        if total == 0:
            return 0.0
        
        w_ada /= total
        w_xgb /= total
        w_cat /= total
        
        blended_probs = (w_ada * oof_probs['adaboost'] + 
                         w_xgb * oof_probs['xgboost'] + 
                         w_cat * oof_probs['catboost'])
        
        preds = (blended_probs >= 0.5).astype(int)
        return f1_score(y, preds)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100)
    
    best_weights = study.best_params
    total = sum(best_weights.values())
    final_weights = [best_weights['w_ada']/total, best_weights['w_xgb']/total, best_weights['w_cat']/total]
    print(f"Optimal Weights -> AdaBoost: {final_weights[0]:.3f}, XGBoost: {final_weights[1]:.3f}, CatBoost: {final_weights[2]:.3f}")
    
    # Calculate final OOF metrics
    blended_probs = (final_weights[0] * oof_probs['adaboost'] + 
                     final_weights[1] * oof_probs['xgboost'] + 
                     final_weights[2] * oof_probs['catboost'])
    preds = (blended_probs >= 0.5).astype(int)
    
    metrics = {
        'F1_Score': f1_score(y, preds),
        'Precision': precision_score(y, preds),
        'Recall': recall_score(y, preds),
        'PR_AUC': average_precision_score(y, blended_probs),
        'Accuracy': accuracy_score(y, preds)
    }
    
    return final_weights, metrics

def log_result(log_path, results):
    row = f"{results['Model_Name']},{results['CV_Accuracy']:.4f},{results['CV_F1_Score']:.4f},{results['CV_PR_AUC']:.4f}," \
          f"{results['CV_Precision']:.4f},{results['CV_Recall']:.4f},\"{str(results['Best_Parameters']).replace('\"','\'')}\"," \
          f"{results['Model_File_Path']},{results['Output_Predictions_Path']}\n"
    with open(log_path, 'a') as f:
        f.write(row)

def main():
    train, test, test_ids, base_dir = load_data()
    X = train.drop(columns=['is_fraud'])
    y = train['is_fraud']
    X_test = test
    
    models = load_models(base_dir)
    oof_probs = get_oof_predictions(models, X, y)
    
    best_weights, ensemble_metrics = optimize_weights(oof_probs, y)
    print(f"Ensemble CV F1-Score: {ensemble_metrics['F1_Score']:.4f}")
    
    # Build VotingClassifier
    estimators = [('adaboost', models['adaboost']), 
                  ('xgboost', models['xgboost']), 
                  ('catboost', models['catboost'])]
                  
    ensemble_model = VotingClassifier(estimators=estimators, voting='soft', weights=best_weights)
    print("Training Final Ensemble Model on full dataset...")
    ensemble_model.fit(X, y)
    
    # Save Ensemble Artifacts
    ens_models_dir = os.path.join(base_dir, "models", "ensemble")
    ens_outputs_dir = os.path.join(base_dir, "outputs", "ensemble")
    os.makedirs(ens_models_dir, exist_ok=True)
    os.makedirs(ens_outputs_dir, exist_ok=True)
    
    model_path = os.path.join(ens_models_dir, "best_model.pkl")
    preds_path = os.path.join(ens_outputs_dir, "predictions.csv")
    
    with open(model_path, 'wb') as f:
        pickle.dump(ensemble_model, f)
        
    print("Predicting on test set...")
    ensemble_preds = ensemble_model.predict(X_test)
    ens_sub = pd.DataFrame({'transaction_id': test_ids['transaction_id'], 'is_fraud': ensemble_preds})
    ens_sub.to_csv(preds_path, index=False)
    
    # Log Ensemble
    results = {
        'Model_Name': 'Ensemble_SoftVoting',
        'CV_Accuracy': ensemble_metrics['Accuracy'],
        'CV_F1_Score': ensemble_metrics['F1_Score'],
        'CV_PR_AUC': ensemble_metrics['PR_AUC'],
        'CV_Precision': ensemble_metrics['Precision'],
        'CV_Recall': ensemble_metrics['Recall'],
        'Best_Parameters': {'w_ada': best_weights[0], 'w_xgb': best_weights[1], 'w_cat': best_weights[2]},
        'Model_File_Path': "models/ensemble/best_model.pkl",
        'Output_Predictions_Path': "outputs/ensemble/predictions.csv"
    }
    
    log_path = os.path.join(base_dir, "outputs", "model_results_log.csv")
    log_result(log_path, results)
    
    # Select Best Model for Final Submission
    best_single_f1 = 0.9957 # AdaBoost
    best_single_preds_path = os.path.join(base_dir, "outputs", "adaboost", "predictions.csv")
    final_sub_path = os.path.join(base_dir, "outputs", "FINAL_SUBMISSION.csv")
    
    print("\n--- Final Model Selection ---")
    if ensemble_metrics['F1_Score'] > best_single_f1:
        print("The Ensemble outperformed the best single model!")
        print("Using Ensemble predictions for FINAL_SUBMISSION.csv")
        ens_sub.to_csv(final_sub_path, index=False)
    else:
        print("The best single model (AdaBoost) outperformed or matched the Ensemble.")
        print("Using AdaBoost predictions for FINAL_SUBMISSION.csv")
        best_preds = pd.read_csv(best_single_preds_path)
        best_preds.to_csv(final_sub_path, index=False)
        
    print(f"Successfully created final submission file at: {final_sub_path}")

if __name__ == "__main__":
    main()


## 8. Phase 7: Advanced Ensemble (Stacking Meta-Learner)

**Source:** `src/modeling/advanced_ensemble.py`

### 8.1 Strategy
We attempted a more advanced 2-level stacking approach using all 5 boosting models:

- **Level-0 (Base Models):** AdaBoost, XGBoost, CatBoost, LightGBM, GradientBoosting
- **Level-1 (Meta-Learner):** Ridge Logistic Regression (`LogisticRegression(penalty='l2', C=1.0)`)

### 8.2 Process

#### Step 1: Generate OOF Meta-Features
Each row becomes a 5-dimensional vector of fraud probabilities from each base model.

#### Step 2: Train Meta-Learner on OOF Features
**Why Logistic Regression?** As documented in our overfitting analysis report, a simple, regularized linear model as the meta-learner prevents overfitting. Using a complex model (like another XGBoost) as the meta-learner would memorize noise.

#### Step 3: Dynamic Threshold Optimization
**Result:** Optimal threshold found at **0.02** (very aggressive — catches maximum fraud).

### 8.3 Advanced Ensemble Results
| Metric | Score |
|---|---|
| **F1-Score** | **0.9959** |
| **Precision** | **1.0000** |
| **Recall** | **0.9917** |
| **PR-AUC** | **0.9990** |

**The advanced stacking ensemble tied the basic soft-voting ensemble** — both achieved F1 = 0.9959. The additional complexity of stacking 5 models through a meta-learner provided zero improvement.

---

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, accuracy_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

def load_data():
    base_dir = r"c:\Users\ADMIN\Desktop\BigBug_Octwave"
    processed_dir = os.path.join(base_dir, "data", "processed")
    train = pd.read_csv(os.path.join(processed_dir, "train_engineered.csv"))
    test = pd.read_csv(os.path.join(processed_dir, "test_engineered.csv"))
    test_ids = pd.read_csv(os.path.join(processed_dir, "test_transaction_ids.csv"))
    return train, test, test_ids, base_dir

def load_models(base_dir):
    models = {}
    top_models = ['adaboost', 'xgboost', 'catboost', 'gradientboosting', 'lightgbm']
    for name in top_models:
        path = os.path.join(base_dir, "models", name, "best_model.pkl")
        with open(path, 'rb') as f:
            models[name] = pickle.load(f)
    return models

def get_oof_meta_features(models, X, y):
    print("Generating OOF Meta-Features (This might take a minute)...")
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    meta_features = np.zeros((X.shape[0], len(models)))
    
    for i, (name, model) in enumerate(models.items()):
        print(f"  Generating OOF for {name}...")
        probs = cross_val_predict(model, X, y, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
        meta_features[:, i] = probs
        
    return meta_features

def optimize_threshold(y_true, y_probs):
    print("Optimizing Classification Threshold...")
    thresholds = np.arange(0.01, 1.0, 0.01)
    best_thresh = 0.5
    best_f1 = 0.0
    
    for thresh in thresholds:
        preds = (y_probs >= thresh).astype(int)
        score = f1_score(y_true, preds)
        if score > best_f1:
            best_f1 = score
            best_thresh = thresh
            
    print(f"Optimal Threshold Found: {best_thresh:.2f} (F1: {best_f1:.4f})")
    return best_thresh

def log_result(log_path, results):
    row = f"{results['Model_Name']},{results['CV_Accuracy']:.4f},{results['CV_F1_Score']:.4f},{results['CV_PR_AUC']:.4f}," \
          f"{results['CV_Precision']:.4f},{results['CV_Recall']:.4f},\"{str(results['Best_Parameters']).replace('\"','\'')}\"," \
          f"{results['Model_File_Path']},{results['Output_Predictions_Path']}\n"
    with open(log_path, 'a') as f:
        f.write(row)

def main():
    train, test, test_ids, base_dir = load_data()
    X = train.drop(columns=['is_fraud'])
    y = train['is_fraud']
    X_test = test
    
    models = load_models(base_dir)
    
    # Generate Level-0 OOF Features
    X_meta = get_oof_meta_features(models, X, y)
    
    # Train Level-1 Meta-Learner on OOF features (L2 Regularized)
    print("Training Level-1 Meta-Learner (Ridge Logistic Regression)...")
    meta_model = LogisticRegression(penalty='l2', C=1.0, random_state=42)
    
    # Get OOF probabilities from Meta-Learner to evaluate ensemble performance
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    meta_oof_probs = cross_val_predict(meta_model, X_meta, y, cv=cv, method='predict_proba')[:, 1]
    
    # Dynamic Threshold Optimization
    best_thresh = optimize_threshold(y, meta_oof_probs)
    
    # Calculate Final Metrics with Best Threshold
    final_cv_preds = (meta_oof_probs >= best_thresh).astype(int)
    metrics = {
        'F1_Score': f1_score(y, final_cv_preds),
        'Precision': precision_score(y, final_cv_preds),
        'Recall': recall_score(y, final_cv_preds),
        'PR_AUC': average_precision_score(y, meta_oof_probs),
        'Accuracy': accuracy_score(y, final_cv_preds)
    }
    print(f"\nAdvanced Stacking Ensemble CV F1-Score: {metrics['F1_Score']:.4f}")
    
    # Train Final Meta-Model on full Level-0 features
    meta_model.fit(X_meta, y)
    
    # Save Advanced Ensemble Artifacts
    ens_models_dir = os.path.join(base_dir, "models", "advanced_ensemble")
    ens_outputs_dir = os.path.join(base_dir, "outputs", "advanced_ensemble")
    os.makedirs(ens_models_dir, exist_ok=True)
    os.makedirs(ens_outputs_dir, exist_ok=True)
    
    model_path = os.path.join(ens_models_dir, "best_model.pkl")
    preds_path = os.path.join(ens_outputs_dir, "predictions.csv")
    
    # Save the meta-learner and the threshold info
    ensemble_package = {
        'base_models': models,
        'meta_model': meta_model,
        'optimal_threshold': best_thresh
    }
    with open(model_path, 'wb') as f:
        pickle.dump(ensemble_package, f)
        
    print("Generating Final Test Predictions...")
    # Generate test features for Meta-Learner
    X_test_meta = np.zeros((X_test.shape[0], len(models)))
    for i, (name, model) in enumerate(models.items()):
        X_test_meta[:, i] = model.predict_proba(X_test)[:, 1]
        
    # Predict using meta-learner and custom threshold
    test_probs = meta_model.predict_proba(X_test_meta)[:, 1]
    final_test_preds = (test_probs >= best_thresh).astype(int)
    
    ens_sub = pd.DataFrame({'transaction_id': test_ids['transaction_id'], 'is_fraud': final_test_preds})
    ens_sub.to_csv(preds_path, index=False)
    
    # Log Advanced Ensemble
    results = {
        'Model_Name': 'Advanced_Stacking_Ensemble',
        'CV_Accuracy': metrics['Accuracy'],
        'CV_F1_Score': metrics['F1_Score'],
        'CV_PR_AUC': metrics['PR_AUC'],
        'CV_Precision': metrics['Precision'],
        'CV_Recall': metrics['Recall'],
        'Best_Parameters': {'meta_model': 'LogisticRegression_L2', 'optimal_threshold': best_thresh},
        'Model_File_Path': "models/advanced_ensemble/best_model.pkl",
        'Output_Predictions_Path': "outputs/advanced_ensemble/predictions.csv"
    }
    
    log_path = os.path.join(base_dir, "outputs", "model_results_log.csv")
    log_result(log_path, results)
    
    # Final Submission Selection
    best_single_f1 = 0.9959 # Previous Soft-Voting Ensemble
    final_sub_path = os.path.join(base_dir, "outputs", "FINAL_SUBMISSION.csv")
    
    print("\n--- Final Model Selection ---")
    if metrics['F1_Score'] > best_single_f1:
        print("The Advanced Stacking Ensemble outperformed the basic Soft-Voting Ensemble!")
        print("Overwriting FINAL_SUBMISSION.csv with new advanced predictions.")
        ens_sub.to_csv(final_sub_path, index=False)
    else:
        print("The basic Soft-Voting Ensemble matched or outperformed the Stacking Meta-Learner.")
        print("Keeping the previous FINAL_SUBMISSION.csv intact.")
        
    print("Advanced Execution Complete!")

if __name__ == "__main__":
    main()


## 9. Phase 8: Final Model Selection & Submission

**Source:** `src/modeling/ensemble.py` (lines 151-167) and `src/modeling/advanced_ensemble.py` (lines 147-160)

### 9.1 Selection Logic

Both scripts contain explicit model selection logic:

**In `ensemble.py`:**
**Result:** Ensemble (0.9959) > AdaBoost (0.9957) → **Ensemble predictions saved to `FINAL_SUBMISSION.csv`**

**In `advanced_ensemble.py`:**
**Result:** Advanced (0.9959) is NOT > basic ensemble (0.9959) → **Previous `FINAL_SUBMISSION.csv` kept intact**

### 9.2 Final Decision
The **3-model Soft Voting Ensemble** (AdaBoost + XGBoost + CatBoost) was selected as the final submission model because:
1. It achieved the highest F1-Score (0.9959)
2. The more complex 5-model stacking approach failed to beat it
3. Simpler models generalize better on unseen private leaderboard data (Occam's Razor)

### 9.3 Pipeline Execution Order
**Source:** `src/run_pipeline.py`

The entire pipeline was orchestrated by a single runner script:
**Note:** `src/inference/predict.py` is an empty file — predictions were already generated within the ensemble scripts themselves.

---

## 10. Final Evaluation, Visualization & Submission Strategy

**Source:** `notebooks/04_Evaluation_and_Prediction.ipynb`

### 10.1 Evaluation Visualizations
To thoroughly validate our final models, the following diagnostic plots were generated on the Out-Of-Fold predictions:
- **Confusion Matrices** for the top models (AdaBoost and the Ensemble)
- **ROC Curves** (Receiver Operating Characteristic)
- **Precision-Recall (PR) Curves** (Crucial for severe class imbalance)
- **Threshold Sweep Plot** (F1, Precision, Recall plotted against threshold values from 0.01 to 0.99)
- **Cross-Model Feature Importance** (Bar charts showing normalized importance ranking averaged across models)

### 10.2 Dual Submission Strategy
Based on the threshold sweep, two distinct submission strategies were explored:
- **Standard Submission (Threshold = 0.5):** The default decision boundary, achieving perfect precision.
- **Aggressive Submission (Threshold = 0.3):** A lower threshold that maximizes recall at the cost of precision, designed to catch borderline fraud cases that the 0.5 threshold might miss.

### 10.3 Submission Validation Logic
Before saving `submission.csv`, a validation function was applied to ensure strict compliance with Kaggle's expected format. It checked:
1. **Shape match:** Exactly matches `sample_submission.csv` (2000 rows).
2. **Column names:** Exactly `transaction_id` and `is_fraud`.
3. **Data types:** Both columns must be integers.
4. **Value range:** `is_fraud` must contain only 0s and 1s.
5. **Transaction IDs:** Must perfectly match the test set IDs.

---

## 11. Final Results Summary

### The Winning Model
**Optuna-Weighted 3-Model Soft Voting Ensemble**
- AdaBoost (weight: 0.284)
- XGBoost (weight: 0.423)
- CatBoost (weight: 0.292)
- Decision threshold: 0.5

### Final OOF Metrics
| Metric | Value |
|---|---|
| **F1-Score** | **0.9959** |
| **Precision** | **1.0000** |
| **Recall** | **0.9917** |
| **PR-AUC** | **0.9991** |
| **Accuracy** | **0.9999** |

### What This Means in Practice
- **Precision = 1.0000:** Every single transaction our model flagged as fraud WAS actually fraud. Zero false alarms.
- **Recall = 0.9917:** Out of 121 fraudulent transactions in training, we correctly identified 120. We missed exactly **1** fraud case.
- **F1 = 0.9959:** The harmonic mean of perfect precision and near-perfect recall.

### Output Files
| File | Location | Description |
|---|---|---|
| Final submission | `outputs/FINAL_SUBMISSION.csv` | The CSV uploaded to Kaggle |
| Ensemble model | `models/ensemble/best_model.pkl` | Serialized VotingClassifier |
| All model results | `outputs/model_results_log.csv` | Central log of all 12 models + 2 ensembles |

---

*Team BigBug — OctWave 3.0 Credit Card Fraud Detection Challenge, August 2026*